# SezimBand — Section 2.2: reproducible model development

Готовая самостоятельная версия на основе присланного Kaggle-кода. Загрузите в Kaggle, подключите UrbanSound8K, включите GPU и выполните **Run All**. Старые ячейки запускать не нужно. Если переносите код в прежний ноутбук, замените preprocessing, split и training соответствующими блоками отсюда; нельзя просто присоединить корректный тест к уже обученной на нём модели.

**Сохранено:** 10 классов, MFCC 40×173, исходная CNN, Adam 0.001, batch size 8, максимум 15 epochs, отсутствие аугментаций. **Изменено:** параметры preprocessing явно заданы; folds 1–8 / 9 / 10; контроль общих fsID; восстановление лучшей эпохи; сравнение с TinyCNN; экспорт данных документации.

Это **один holdout по predefined folds**, не 10-fold cross-validation. Новые результаты нельзя подменять опубликованными ~91% из исходного random split. Обучение здесь ещё не выполнено; все числа результатов появятся после запуска.

В исходном подходе нет фиксированных аудиоокон: каждый файл даёт один MFCC-пример, а временная ось интерполируется до 173 столбцов. Поэтому текст о фиксированных окнах нужно заменить автоматически сформированным описанием. 40 MFCC — не 40 mel-полос: здесь явно используются 128 mel-полос.

Источники: [исходный Kaggle notebook](https://www.kaggle.com/code/bhavybansal/urbansound8k-cnn-classifier), [UrbanSound8K](https://urbansounddataset.weebly.com/urbansound8k.html), [librosa MFCC](https://librosa.org/doc/latest/generated/librosa.feature.mfcc.html). Укажите авторство исходного кода в работе.

Local run note: this copy uses local dataset/output paths and excludes source-recording overlap before preprocessing; see source_overlap_resolution.json.


## 1. Imports and configuration
Обычно зависимости уже есть в Kaggle. При ошибке импорта выполните в отдельной ячейке `%pip install tensorflow librosa soundfile opencv-python-headless pandas matplotlib scikit-learn tqdm resampy`, затем перезапустите kernel. Для установки нужен Internet.
Параметры ниже относятся к новому запуску; они не доказывают версию/defaults исторического запуска 2022 года.


In [ ]:
from pathlib import Path
import os, json, platform, importlib.metadata, time, shutil
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import cv2
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from tqdm.auto import tqdm
from IPython.display import display, FileLink

SEED = 42
keras.utils.set_random_seed(SEED)
# Set an exact CSV path if automatic discovery finds multiple datasets.
METADATA_CSV = 'C:\\Users\\dimat\\OneDrive\\Desktop\\sezimband_22\\UrbanSound8K\\metadata\\UrbanSound8K.csv'
SEARCH_ROOT = Path('C:\\Users\\dimat\\OneDrive\\Desktop\\sezimband_22\\UrbanSound8K\\metadata')
OUT = Path('C:\\Users\\dimat\\OneDrive\\Desktop\\sezimband_22\\sezimband_2_2')
OUT.mkdir(parents=True, exist_ok=True)
CFG = dict(sr=22050, mono=True, res_type='kaiser_fast', n_fft=2048,
           win_length=2048, hop_length=512, window='hann', center=True,
           pad_mode='constant', power=2.0, n_mels=128, fmin=0.0,
           fmax=11025.0, htk=False, mel_norm='slaney', n_mfcc=40,
           dct_type=2, dct_norm='ortho', lifter=0,
           db_ref=1.0, db_amin=1e-10, db_top=80.0,
           resized_frames=173, resize='cv2.INTER_LINEAR',
           segmentation='one complete excerpt per example; no fixed windows',
           feature_standardization='none', augmentations=[])
TRAIN_FOLDS, VAL_FOLDS, TEST_FOLDS = list(range(1,9)), [9], [10]
BATCH_SIZE, MAX_EPOCHS, PATIENCE, LEARNING_RATE = 8, 15, 8, 0.001
ALERT_CLASSES = ['car_horn', 'siren']

def save_json(name, data):
    (OUT / name).write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding='utf-8')

def save_table(frame, stem):
    frame.to_csv(OUT / (stem + '.csv'), index=False)
    frame.to_html(OUT / (stem + '.html'), index=False, float_format=lambda v: f'{v:.4f}')
    display(frame)

def save_figure(fig, stem):
    fig.savefig(OUT / (stem + '.png'), dpi=300, bbox_inches='tight')
    fig.savefig(OUT / (stem + '.svg'), bbox_inches='tight')
    plt.show()
    plt.close(fig)

versions = {'python': platform.python_version()}
for package in ['tensorflow', 'keras', 'librosa', 'soundfile', 'numpy', 'pandas', 'scikit-learn', 'opencv-python-headless', 'opencv-python', 'resampy']:
    try: versions[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError: pass
save_json('environment.json', versions)
save_json('feature_config.json', CFG)
print('GPU:', tf.config.list_physical_devices('GPU'))
print('Output:', OUT)


## 2. Metadata, audio audit and feature extraction
Таблица 2.2 различает уникальные исходные записи **fsID**, файлы-фрагменты и примеры модели. Сумма длительностей фрагментов включает возможные перекрытия; дополнительно считается объединение интервалов внутри каждого fsID. Это не полная длительность исходных Freesound-записей.
Проверяется каждый файл; при ошибке выполнение остановится с отчётом, данные не исключаются молча.


In [ ]:
if METADATA_CSV is None:
    candidates = sorted(SEARCH_ROOT.rglob('UrbanSound8K.csv'))
    if len(candidates) != 1:
        raise ValueError(f'Set METADATA_CSV to one exact path. Found: {candidates}')
    metadata_path = candidates[0]
else:
    metadata_path = Path(METADATA_CSV)
df = pd.read_csv(metadata_path).reset_index(drop=True)
original_rows = len(df)
_partition = np.select(
    [df.fold.isin(TRAIN_FOLDS), df.fold.isin(VAL_FOLDS), df.fold.isin(TEST_FOLDS)],
    ['train', 'validation', 'test'], default='unknown')
_rank = pd.Series(_partition, index=df.index).map({'train': 0, 'validation': 1, 'test': 2})
if _rank.isna().any():
    raise ValueError('Unexpected fold in UrbanSound8K metadata')
_highest_rank = _rank.groupby(df.fsID).transform('max')
excluded_overlap = df.loc[_rank < _highest_rank].copy()
excluded_overlap['original_partition'] = _partition[_rank < _highest_rank]
excluded_overlap.to_csv(OUT / 'excluded_source_overlap.csv', index=False)
df = df.loc[_rank == _highest_rank].reset_index(drop=True)
save_json('source_overlap_resolution.json', {
    'original_excerpt_count': int(original_rows),
    'retained_excerpt_count': int(len(df)),
    'excluded_excerpt_count': int(len(excluded_overlap)),
    'excluded_fsIDs': sorted(int(x) for x in excluded_overlap.fsID.unique()),
    'rule': 'For fsID crossing partitions, retain only the highest-priority partition: test > validation > train.',
    'details_csv': 'excluded_source_overlap.csv',
})
print(f'Excluded {len(excluded_overlap)} overlapping source excerpts from {original_rows} originals')
required = {'slice_file_name','fsID','start','end','fold','classID','class'}
assert required.issubset(df.columns), f'Missing columns: {required-set(df.columns)}'
assert len(df) > 0 and not df[list(required)].isna().any().any()
assert not df.slice_file_name.duplicated().any()
assert set(df.fold) == set(range(1,11))
assert (df.end > df.start).all()
assert df.groupby('classID')['class'].nunique().max() == 1
assert df.groupby('class')['classID'].nunique().max() == 1
CLASSES = sorted(df['class'].unique().tolist())  # same ordering as LabelEncoder
assert len(CLASSES) == 10 and set(ALERT_CLASSES).issubset(CLASSES)
label_to_index = {name: i for i, name in enumerate(CLASSES)}
save_json('class_mapping.json', {'classes': CLASSES, 'alert_classes': ALERT_CLASSES,
                              'non_alert_classes': [c for c in CLASSES if c not in ALERT_CLASSES]})
# Covers common Kaggle layouts and the official audio/metadata layout.
roots = [metadata_path.parent, metadata_path.parent / 'audio',
         metadata_path.parent.parent / 'audio', metadata_path.parent / 'UrbanSound8K' / 'audio']
first = df.iloc[0]
audio_roots = [p for p in roots if (p / f'fold{first.fold}' / first.slice_file_name).is_file()]
if not audio_roots:
    raise FileNotFoundError('Cannot locate fold folders. Edit roots in this cell to point to your audio directory.')
AUDIO_ROOT = audio_roots[0]
df['path'] = [str(AUDIO_ROOT / f'fold{r.fold}' / r.slice_file_name) for r in df.itertuples()]

def extract_features(path):
    audio, sr = librosa.load(path, sr=CFG['sr'], mono=CFG['mono'], res_type=CFG['res_type'], dtype=np.float32)
    if audio.size == 0 or not np.isfinite(audio).all():
        raise ValueError('Empty/non-finite audio')
    mel = librosa.feature.melspectrogram(y=audio, sr=sr,
        n_fft=CFG['n_fft'], hop_length=CFG['hop_length'], win_length=CFG['win_length'],
        window=CFG['window'], center=CFG['center'], pad_mode=CFG['pad_mode'],
        power=CFG['power'], n_mels=CFG['n_mels'], fmin=CFG['fmin'], fmax=CFG['fmax'],
        htk=CFG['htk'], norm=CFG['mel_norm'])
    logmel = librosa.power_to_db(mel, ref=CFG['db_ref'], amin=CFG['db_amin'], top_db=CFG['db_top'])
    mfcc = librosa.feature.mfcc(S=logmel, n_mfcc=CFG['n_mfcc'],
                               dct_type=CFG['dct_type'], norm=CFG['dct_norm'], lifter=CFG['lifter'])
    resized = cv2.resize(mfcc, (CFG['resized_frames'], CFG['n_mfcc']), interpolation=cv2.INTER_LINEAR)
    if not np.isfinite(resized).all(): raise ValueError('Non-finite features')
    return audio, mfcc, resized.astype(np.float32)

X = np.empty((len(df), CFG['n_mfcc'], CFG['resized_frames'], 1), dtype=np.float32)
audit, errors = [], []
for i, row in enumerate(tqdm(df.itertuples(index=False), total=len(df), desc='Audio and MFCC')):
    try:
        info = sf.info(row.path)
        audio, mfcc, resized = extract_features(row.path)
        X[i, :, :, 0] = resized
        audit.append(dict(row_index=i, native_sr=info.samplerate, native_channels=info.channels,
                          duration_s=info.frames/info.samplerate, native_mfcc_frames=mfcc.shape[1],
                          resampled_samples=len(audio)))
    except Exception as exc:
        errors.append({'file':row.path, 'error':str(exc)})
if errors:
    pd.DataFrame(errors).to_csv(OUT/'audio_errors.csv', index=False)
    raise RuntimeError(f'{len(errors)} audio errors. See audio_errors.csv; fix before continuing.')
df = df.join(pd.DataFrame(audit).set_index('row_index'))
df['model_examples'] = 1
y_indices = df['class'].map(label_to_index).to_numpy(dtype=np.int64)
y = keras.utils.to_categorical(y_indices, len(CLASSES))
print('X:', X.shape, 'y:', y.shape)


In [ ]:
def union_duration(group):
    total = 0.0
    for _, recording in group.groupby('fsID'):
        intervals = sorted(zip(recording.start, recording.end))
        left, right = intervals[0]
        for a, b in intervals[1:]:
            if a <= right: right = max(right, b)
            else:
                total += right-left
                left, right = a, b
        total += right-left
    return total

rows = []
for cls, g in df.groupby('class', sort=True):
    rows.append(dict(class_name=cls, source='UrbanSound8K',
        alert=cls in ALERT_CLASSES, original_recordings_fsID=g.fsID.nunique(),
        excerpt_files=len(g), summed_excerpt_duration_s=g.duration_s.sum(),
        unique_annotated_coverage_s=union_duration(g), model_examples=int(g.model_examples.sum()),
        fixed_window_count='not applicable'))
rows.append(dict(class_name='TOTAL', source='UrbanSound8K', alert='mixed',
    original_recordings_fsID=df.fsID.nunique(), excerpt_files=len(df),
    summed_excerpt_duration_s=df.duration_s.sum(), unique_annotated_coverage_s=union_duration(df),
    model_examples=len(df), fixed_window_count='not applicable'))
table22 = pd.DataFrame(rows)
save_table(table22, 'table_2_2_dataset_composition')


## 3. Figure 2.3 — waveform and the exact model input
Пример выбирается из train fold до обучения. Нижняя ось — нормированное положение внутри фрагмента, поскольку resize изменяет временную шкалу. Исходный шаг MFCC не равен шагу после resize.


In [ ]:
example = df[(df['class']=='siren') & df.fold.isin(TRAIN_FOLDS)].iloc[0]
audio, native_mfcc, resized = extract_features(example.path)
fig, ax = plt.subplots(2, 1, figsize=(10, 6), constrained_layout=True)
ax[0].plot(np.arange(len(audio))/CFG['sr'], audio, linewidth=0.6, color='#24608b')
ax[0].set(xlabel='Time (s)', ylabel='Amplitude', title=f'Siren waveform — {example.slice_file_name}')
im = ax[1].imshow(resized, origin='lower', aspect='auto', extent=[0,1,-0.5,39.5], cmap='magma')
ax[1].set(xlabel='Relative position within excerpt (after interpolation)', ylabel='MFCC coefficient index',
          title='Actual model input: 40 MFCC coefficients × 173 resized frames')
fig.colorbar(im, ax=ax[1], label='MFCC value')
save_figure(fig, 'figure_2_3_waveform_mfcc')
save_json('figure_2_3_source.json', {'file':example.slice_file_name, 'fold':int(example.fold),
    'duration_s':len(audio)/CFG['sr'], 'native_frames':int(native_mfcc.shape[1])})


## 4. Table 2.3 — independent train / validation / test
Folds 1–8 → training, fold 9 → validation, fold 10 → final test. Номера фиксируются до результатов. При общих fsID код останавливается. Собственных записей устройства в этом эксперименте нет; их результаты не заявляются.


In [ ]:
fold_sets = [set(TRAIN_FOLDS), set(VAL_FOLDS), set(TEST_FOLDS)]
assert not (fold_sets[0]&fold_sets[1] or fold_sets[0]&fold_sets[2] or fold_sets[1]&fold_sets[2])
assert set.union(*fold_sets) == set(range(1,11))
df['split'] = np.select([df.fold.isin(TRAIN_FOLDS), df.fold.isin(VAL_FOLDS), df.fold.isin(TEST_FOLDS)],
                        ['train','validation','test'], default='unassigned')
assert df.groupby('fsID').split.nunique().max() == 1, 'Related source recordings cross split boundaries!'
indices = {s: np.flatnonzero(df.split.to_numpy()==s) for s in ['train','validation','test']}
for s, ids in indices.items():
    assert set(y_indices[ids]) == set(range(len(CLASSES))), f'Missing classes in {s}'
rows = []
for s, g in df.groupby('split', sort=False):
    rows.append(dict(split=s, folds=','.join(map(str, sorted(g.fold.unique()))),
        excerpt_files=len(g), original_recordings_fsID=g.fsID.nunique(),
        model_examples=len(g), summed_excerpt_duration_s=g.duration_s.sum(),
        purpose={'train':'weight fitting','validation':'checkpoint and architecture selection','test':'final evaluation only'}[s]))
table23 = pd.DataFrame(rows)
save_table(table23, 'table_2_3_data_allocation')
save_table(pd.crosstab(df['class'], df['split']).reset_index(), 'split_class_counts')
df.to_csv(OUT/'audio_manifest.csv', index=False)
save_json('split_protocol.json', {'train_folds':TRAIN_FOLDS, 'validation_folds':VAL_FOLDS,
    'test_folds':TEST_FOLDS, 'source_overlap_check':'passed',
    'evaluation':'single predefined-fold holdout; not 10-fold CV', 'device_recordings_evaluated':False})


## 5. Two architectures
BaselineCNN повторяет слои присланного ноутбука. TinyCNN использует меньше фильтров. Обе модели получают одинаковые данные и условия обучения. MACs ниже — аналитическая оценка Conv2D/Dense для одного примера; она не включает MFCC, pooling, activation и другую работу. Это не время работы ESP32 и не оценка tensor arena.


In [ ]:
INPUT_SHAPE = (CFG['n_mfcc'], CFG['resized_frames'], 1)
def build_baseline():
    return keras.Sequential([
        keras.Input(shape=INPUT_SHAPE),
        layers.Conv2D(32, (3,3), activation='relu', padding='valid'),
        layers.MaxPooling2D(2, padding='same'),
        layers.Conv2D(128, (3,3), activation='relu', padding='valid'),
        layers.MaxPooling2D(2, padding='same'), layers.Dropout(0.3),
        layers.Conv2D(128, (3,3), activation='relu', padding='valid'),
        layers.MaxPooling2D(2, padding='same'), layers.Dropout(0.3),
        layers.GlobalAveragePooling2D(), layers.Dense(256, activation='relu'),
        layers.Dense(len(CLASSES), activation='softmax')], name='BaselineCNN')

def build_tiny():
    return keras.Sequential([
        keras.Input(shape=INPUT_SHAPE),
        layers.Conv2D(8, 3, activation='relu', padding='valid'),
        layers.MaxPooling2D(2, padding='same'),
        layers.Conv2D(16, 3, activation='relu', padding='valid'),
        layers.MaxPooling2D(2, padding='same'),
        layers.Conv2D(32, 3, activation='relu', padding='valid'),
        layers.GlobalAveragePooling2D(), layers.Dropout(0.3),
        layers.Dense(len(CLASSES), activation='softmax')], name='TinyCNN')

def conv_dense_macs(model):
    total = 0
    for layer in model.layers:
        if isinstance(layer, layers.Conv2D):
            h,w,c = map(int, layer.output.shape[1:])
            kh,kw,cin,_ = map(int, layer.kernel.shape)
            total += h*w*c*kh*kw*cin
        elif isinstance(layer, layers.Dense):
            total += int(np.prod(layer.kernel.shape))
    return total

builders = {'BaselineCNN':build_baseline, 'TinyCNN':build_tiny}
for name, builder in builders.items():
    model = builder()
    model.summary()
    if name == 'BaselineCNN': assert model.count_params() == 220490


## 6. Train, restore best checkpoints, compare on validation
Здесь заново обучаются две модели. Максимум 15 epochs, batch size 8, early stopping patience 8, выбор эпохи по минимальной validation loss. Аугментации и нормализация признаков не применяются. Dropout — регуляризация, не аугментация.
Финальная архитектура выбирается по validation loss; размер/MACs публикуются отдельно. Это выбор компьютерной модели для следующего этапа, а не подтверждение пригодности для микроконтроллера.


In [ ]:
trained, histories, comparison = {}, {}, []
tr, va = indices['train'], indices['validation']
for name, builder in builders.items():
    keras.utils.set_random_seed(SEED)
    model = builder()
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
                  loss='categorical_crossentropy', metrics=['accuracy'])
    checkpoint = OUT / f'{name}_best.keras'
    callbacks = [keras.callbacks.ModelCheckpoint(str(checkpoint), monitor='val_loss', mode='min', save_best_only=True),
                 keras.callbacks.EarlyStopping(monitor='val_loss', mode='min', patience=PATIENCE, restore_best_weights=True)]
    started = time.perf_counter()
    history = model.fit(X[tr], y[tr], validation_data=(X[va], y[va]),
                        batch_size=BATCH_SIZE, epochs=MAX_EPOCHS, shuffle=True, callbacks=callbacks, verbose=2)
    training_seconds = time.perf_counter()-started
    model = keras.models.load_model(checkpoint)
    trained[name], histories[name] = model, history.history
    history_frame = pd.DataFrame(history.history)
    history_frame.insert(0, 'epoch', np.arange(1, len(history_frame)+1))
    history_frame.to_csv(OUT/f'{name}_training_curve.csv', index=False)
    val = model.evaluate(X[va], y[va], verbose=0, return_dict=True)
    pred = np.argmax(model.predict(X[va], verbose=0), axis=1)
    (OUT/f'{name}_architecture.json').write_text(model.to_json(), encoding='utf-8')
    summary_lines=[]
    model.summary(print_fn=lambda line, **kwargs: summary_lines.append(line))
    (OUT/f'{name}_summary.txt').write_text('\n'.join(summary_lines), encoding='utf-8')
    tflite_bytes, conversion_error = None, None
    try:
        converter = tf.lite.TFLiteConverter.from_keras_model(model)
        tflite = converter.convert()  # float32; not INT8
        (OUT/f'{name}_float32.tflite').write_bytes(tflite)
        tflite_bytes = len(tflite)
    except Exception as exc:
        conversion_error = str(exc)
        (OUT/f'{name}_tflite_error.txt').write_text(conversion_error, encoding='utf-8')
    comparison.append(dict(model=name, parameters=model.count_params(),
        raw_weight_bytes=sum(w.numpy().nbytes for w in model.weights),
        keras_checkpoint_bytes=checkpoint.stat().st_size, float32_tflite_bytes=tflite_bytes,
        estimated_conv_dense_MACs=conv_dense_macs(model),
        best_epoch=int(np.argmin(history.history['val_loss'])+1), epochs_run=len(history_frame),
        validation_loss=float(val['loss']), validation_accuracy=float(val['accuracy']),
        validation_macro_f1=float(f1_score(y_indices[va], pred, average='macro', zero_division=0)),
        training_seconds=training_seconds, tflite_status='failed' if conversion_error else 'converted',
        esp32_latency_ms='not measured', esp32_peak_ram_bytes='not measured'))
comparison = pd.DataFrame(comparison)
selected_name = comparison.sort_values(['validation_loss','parameters']).iloc[0]['model']
comparison['selected_by_validation_loss'] = comparison.model.eq(selected_name)
save_table(comparison, 'table_2_4_model_comparison')
print('Selected:', selected_name)
save_json('training_config.json', dict(framework='TensorFlow/Keras', loss='categorical_crossentropy',
    optimizer='Adam', learning_rate=LEARNING_RATE, batch_size=BATCH_SIZE, max_epochs=MAX_EPOCHS,
    patience=PATIENCE, seed=SEED, augmentations=[], feature_standardization='none',
    checkpoint_selection='minimum validation loss', architecture_selection='minimum validation loss; parameters break exact ties',
    selected_model=selected_name, input_shape=list(INPUT_SHAPE)))


## 7. Figure 2.4 — training and validation loss
Строится по реальному History каждого запуска. Маркер отмечает восстановленную эпоху.


In [ ]:
fig, axes = plt.subplots(1, len(histories), figsize=(12,4), constrained_layout=True, squeeze=False)
for ax, (name, h) in zip(axes.flat, histories.items()):
    epochs = np.arange(1, len(h['loss'])+1)
    best_epoch = int(np.argmin(h['val_loss'])+1)
    ax.plot(epochs, h['loss'], label='Training')
    ax.plot(epochs, h['val_loss'], label='Validation')
    ax.axvline(best_epoch, color='gray', linestyle='--', label=f'Best epoch: {best_epoch}')
    ax.set(xlabel='Epoch', ylabel='Categorical cross-entropy', title=name)
    ax.legend(); ax.grid(alpha=0.2)
save_figure(fig, 'figure_2_4_training_validation_loss')


## 8. Final test — only the model selected on validation
Запустите после фиксации архитектуры. Не подбирайте настройки по этому тесту. Это результаты на UrbanSound8K, а не на записях браслета. Alert означает: предсказанный argmax-класс равен siren или car_horn; порог вероятности здесь не подбирается.


In [ ]:
te = indices['test']
selected_model = trained[selected_name]
prob = selected_model.predict(X[te], verbose=0)
pred = prob.argmax(axis=1)
true = y_indices[te]
report = classification_report(true, pred, labels=list(range(len(CLASSES))), target_names=CLASSES,
                               output_dict=True, zero_division=0)
save_table(pd.DataFrame(report).T.rename_axis('class_or_metric').reset_index(), 'test_classification_report')
alert_ids = [label_to_index[c] for c in ALERT_CLASSES]
true_alert, pred_alert = np.isin(true, alert_ids), np.isin(pred, alert_ids)
tn, fp, fn, tp = confusion_matrix(true_alert, pred_alert, labels=[False,True]).ravel()
metrics = {'selected_model':selected_name, 'test_fold':TEST_FOLDS,
    'test_examples':len(te), 'accuracy':float(accuracy_score(true,pred)),
    'macro_f1':float(f1_score(true,pred,average='macro',zero_division=0)),
    'alert_precision':float(tp/(tp+fp)) if tp+fp else None,
    'alert_recall':float(tp/(tp+fn)) if tp+fn else None,
    'non_target_false_positive_fraction':float(fp/(fp+tn)) if fp+tn else None,
    'alert_confusion':{'tn':int(tn),'fp':int(fp),'fn':int(fn),'tp':int(tp)},
    'scope':'single UrbanSound8K holdout; no prototype-microphone test'}
save_json('test_metrics.json', metrics)
predictions = df.iloc[te][['slice_file_name','fsID','fold','class']].reset_index(drop=True)
predictions['predicted_class'] = [CLASSES[i] for i in pred]
for i,c in enumerate(CLASSES): predictions['p_'+c] = prob[:,i]
predictions.to_csv(OUT/'holdout_predictions.csv', index=False)
cm = confusion_matrix(true,pred,labels=list(range(len(CLASSES))))
pd.DataFrame(cm,index=CLASSES,columns=CLASSES).to_csv(OUT/'test_confusion_matrix.csv')
fig, ax = plt.subplots(figsize=(10,8), constrained_layout=True)
im=ax.imshow(cm,cmap='Blues')
ax.set(xticks=np.arange(len(CLASSES)), yticks=np.arange(len(CLASSES)),
       xticklabels=CLASSES, yticklabels=CLASSES, xlabel='Predicted class', ylabel='True class',
       title=f'{selected_name}: UrbanSound8K fold {TEST_FOLDS}')
plt.setp(ax.get_xticklabels(),rotation=60,ha='right')
for i in range(len(CLASSES)):
    for j in range(len(CLASSES)):
        ax.text(j,i,str(cm[i,j]),ha='center',va='center',color='white' if cm[i,j]>cm.max()/2 else 'black',fontsize=8)
fig.colorbar(im, ax=ax)
save_figure(fig, 'test_confusion_matrix')
display(metrics)


## 9. Ready-to-use Section 2.2 text and complete export
Создаётся английский текст с параметрами и полученными значениями, а также ZIP. В Table 2.4 размер `.keras` может включать состояние optimiser; для deployment предпочтительнее размер TFLite. Даже успешная конвертация не доказывает запуск на XIAO.
Если TFLite export не сработал, обучение и таблицы сохраняются, а ошибка попадает в отдельный файл.


In [ ]:
selected_row = comparison.set_index('model').loc[selected_name]
non_alert = ', '.join(c for c in CLASSES if c not in ALERT_CLASSES)
text = f"""# 2.2 Audio dataset preparation and model development

The classifier was developed using UrbanSound8K. The original metadata contained {original_rows} excerpts; {len(excluded_overlap)} training excerpts whose source recording also appeared in validation were excluded to prevent leakage. The dataset used in this run contained {len(df)} excerpts from {df.fsID.nunique()} unique source-recording identifiers (fsID). Table 2.2 distinguishes original source recordings, distributed excerpt files, and model examples. Summed excerpt duration may include overlapping fragments; unique annotated coverage is reported separately. No prototype-microphone recordings were evaluated in this experiment.

[Insert Table 2.2: table_2_2_dataset_composition.csv]

Audio was decoded as mono floating-point samples and resampled to {CFG['sr']} Hz using {CFG['res_type']}. Each complete excerpt produced one model example; no fixed-duration audio-window segmentation was performed. A {CFG['n_fft']}-point STFT with a {CFG['win_length']}-sample Hann window and a {CFG['hop_length']}-sample hop ({1000*CFG['hop_length']/CFG['sr']:.4f} ms) was used, with centred frames and constant padding. Power spectra were projected onto {CFG['n_mels']} Slaney-normalised mel bands spanning {CFG['fmin']:.0f}–{CFG['fmax']:.0f} Hz. Log-power values used reference {CFG['db_ref']}, amin {CFG['db_amin']} and an {CFG['db_top']:.0f} dB range. A type-II orthonormal DCT produced {CFG['n_mfcc']} MFCC coefficients with no liftering. The MFCC time axis was linearly interpolated to {CFG['resized_frames']} columns using OpenCV. Thus the model input was 40 × 173 × 1; the resized columns do not represent a fixed physical frame step across different excerpt durations. No feature standardisation was applied.

[Insert Figure 2.3: figure_2_3_waveform_mfcc.png]

The ten output classes were {', '.join(CLASSES)}. An alert label was associated with car_horn or siren under an argmax decision rule. The remaining classes ({non_alert}) were non-alert outputs and were not merged into one training class. These classes do not exhaust possible background sounds encountered by a wrist-worn device.

Training used predefined folds {TRAIN_FOLDS}; validation used fold {VAL_FOLDS}; the final test used fold {TEST_FOLDS}. An explicit check confirmed that no fsID crossed these split boundaries. Validation determined checkpoint and architecture selection. This experiment is a single predefined-fold holdout, not ten-fold cross-validation.

[Insert Table 2.3: table_2_3_data_allocation.csv]

BaselineCNN reproduced the supplied architecture: three valid 3 × 3 convolutions with 32, 128 and 128 filters, three 2 × 2 max-pooling stages with same padding, two dropout layers (0.3), global average pooling, a 256-unit ReLU layer and a ten-unit softmax output. TinyCNN used valid 3 × 3 convolutions with 8, 16 and 32 filters, two same-padded 2 × 2 max-pooling stages, global average pooling, dropout (0.3) and a ten-unit softmax output. All convolutional activations were ReLU. Both models received the same features and data split. Table 2.4 reports parameter counts, measured file sizes, and analytical Conv2D/Dense MAC estimates per example. These estimates exclude feature extraction and other operations and do not measure device latency or peak RAM.

[Insert Table 2.4: table_2_4_model_comparison.csv]

Training used TensorFlow {versions.get('tensorflow','unknown')} / Keras, categorical cross-entropy and Adam with learning rate {LEARNING_RATE}. Batch size was {BATCH_SIZE}, with a maximum of {MAX_EPOCHS} epochs and early-stopping patience {PATIENCE}, monitoring validation loss. No data augmentation was used. The minimum-validation-loss checkpoint was reloaded for each model. {selected_name} was selected by the lower validation loss ({selected_row['validation_loss']:.6f}), with {int(selected_row['parameters']):,} parameters. Its best checkpoint occurred at epoch {int(selected_row['best_epoch'])} of {int(selected_row['epochs_run'])} executed epochs. This selection identifies a candidate for the subsequent embedded feasibility assessment; it does not establish microcontroller compatibility.

[Insert Figure 2.4: figure_2_4_training_validation_loss.png]

After selection, the reserved test fold gave accuracy {metrics['accuracy']:.4f} and macro-F1 {metrics['macro_f1']:.4f}. Class-specific and alert-level results are exported separately for the evaluation section. These are computer-based results on UrbanSound8K and must not be described as achieved by the assembled wristband. Feature settings, software versions, data allocation and model artefacts accompany the export. On-device inference, RAM, latency, microphone transfer and feedback behaviour remain to be tested.
"""
(OUT/'section_2_2_generated.md').write_text(text, encoding='utf-8')
readme = """OUTPUT MAP
Table 2.2 -> table_2_2_dataset_composition.csv / .html
Table 2.3 -> table_2_3_data_allocation.csv / .html
Table 2.4 -> table_2_4_model_comparison.csv / .html
Figure 2.3 -> figure_2_3_waveform_mfcc.png / .svg
Figure 2.4 -> figure_2_4_training_validation_loss.png / .svg
English section -> section_2_2_generated.md
Exact preprocessing -> feature_config.json
Software versions -> environment.json
Training -> training_config.json and *_training_curve.csv
Final test -> test_metrics.json, test_classification_report.csv, holdout_predictions.csv
Raw-weight bytes are not peak inference RAM. Training seconds are not device inference latency.
The current dataset has one example per excerpt; no fixed audio windows.
Model files were trained here only when the notebook was executed; never reuse old random-split scores.
"""
(OUT/'README.txt').write_text(readme, encoding='utf-8')
archive = shutil.make_archive(str(OUT.parent/'sezimband_2_2_results'), 'zip', root_dir=OUT)
print(text)
print('Archive:', archive)
display(FileLink(archive))
